In [119]:
from IPython.core.display import display, HTML
display(HTML("<style>.container { width:100% !important; }</style>"))

/var/folders/db/87d025j5081fm4bs5t49d62r0000gn/T/ipykernel_33405/3777615979.py:1: DeprecationWarning: Importing display from IPython.core.display is deprecated since IPython 7.14, please import from IPython display
  from IPython.core.display import display, HTML


# Lab | Natural Language Processing
### SMS: SPAM or HAM

### Let's prepare the environment

In [120]:
import nltk
import re
import string
import pandas as pd
import matplotlib.pyplot as plt
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import PorterStemmer, WordNetLemmatizer
from nltk import pos_tag
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

- Read Data for the Fraudulent Email Kaggle Challenge
- Reduce the training set to speead up development. 

In [121]:
## Read Data for the Fraudulent Email Kaggle Challenge
data = pd.read_csv("../data/kg_train.csv", encoding="latin-1")

# Reduce the training set to speed up development. 
# Modify for final system
data = data.head(1000)
print(data.shape)
data.fillna("",inplace=True)

(1000, 2)


### Let's divide the training and test set into two partitions

In [122]:
print(data.head())

                                                text  label
0  DEAR SIR, STRICTLY A PRIVATE BUSINESS PROPOSAL...      1
1                                           Will do.      0
2  Nora--Cheryl has emailed dozens of memos about...      0
3  Dear Sir=2FMadam=2C I know that this proposal ...      1
4                                                fyi      0


In [123]:
print(data.columns)

Index(['text', 'label'], dtype='object')


In [124]:
from sklearn.model_selection import train_test_split

data_train, data_val = train_test_split(
    data,
    test_size=0.2,
    random_state=42
)

## Data Preprocessing

In [125]:
import string
from nltk.corpus import stopwords
print(string.punctuation)
print(stopwords.words("english")[100:110])
from nltk.stem.snowball import SnowballStemmer
snowball = SnowballStemmer('english')

!"#$%&'()*+,-./:;<=>?@[\]^_`{|}~
['needn', "needn't", 'no', 'nor', 'not', 'now', 'o', 'of', 'off', 'on']


## Now, we have to clean the html code removing words

- First we remove inline JavaScript/CSS
- Then we remove html comments. This has to be done before removing regular tags since comments can contain '>' characters
- Next we can remove the remaining tags

In [126]:
import re

def clean_html(text):
    # Remove inline JavaScript and CSS
    text = re.sub(r"<(script|style).*?>.*?</\1>", "", text, flags=re.DOTALL)

    # Remove HTML comments
    text = re.sub(r"<!--.*?-->", "", text, flags=re.DOTALL)

    # Remove remaining HTML tags
    text = re.sub(r"<.*?>", "", text)

    return text

In [127]:
data_train["preprocessed_text"] = data_train["text"].apply(clean_html)
data_val["preprocessed_text"] = data_val["text"].apply(clean_html)

- Remove all the special characters
    
- Remove numbers
    
- Remove all single characters
 
- Remove single characters from the start

- Substitute multiple spaces with single space

- Remove prefixed 'b'

- Convert to Lowercase

In [128]:
# Your code

def clean_text(text):
    # Remove special characters
    text = re.sub(r"[^\w\s]", " ", text)

    # Remove numbers
    text = re.sub(r"\d+", "", text)

    # Remove single characters
    text = re.sub(r"\b[a-zA-Z]\b", "", text)

    # Remove single characters from the beginning
    text = re.sub(r"^\s*[a-zA-Z]\s+", "", text)

    # Replace multiple spaces with a single space
    text = re.sub(r"\s+", " ", text)

    # Remove prefixed 'b'
    text = re.sub(r"^b\s+", "", text)

    # Convert to lowercase
    text = text.lower()

    return text

In [129]:
data_train["preprocessed_text"] = data_train["preprocessed_text"].apply(clean_text)
data_val["preprocessed_text"] = data_val["preprocessed_text"].apply(clean_text)

## Now let's work on removing stopwords
Remove the stopwords.

In [130]:
from nltk.corpus import stopwords

stop_words = set(stopwords.words("english"))

def remove_stopwords(text):
    words = text.split()

    filtered_words = []

    for word in words:
        if word not in stop_words:
            filtered_words.append(word)

    return " ".join(filtered_words)


data_train["preprocessed_text"] = data_train["preprocessed_text"].apply(remove_stopwords)
data_val["preprocessed_text"] = data_val["preprocessed_text"].apply(remove_stopwords)

## Tame Your Text with Lemmatization
Break sentences into words, then use lemmatization to reduce them to their base form (e.g., "running" becomes "run"). See how this creates cleaner data for analysis!

In [131]:
from nltk.stem import WordNetLemmatizer

lemmatizer = WordNetLemmatizer()

def lemmatize_text(text):
    words = text.split()

    lemmatized_words = []

    for word in words:
        lemmatized_words.append(lemmatizer.lemmatize(word))

    return " ".join(lemmatized_words)


data_train["preprocessed_text"] = data_train["preprocessed_text"].apply(lemmatize_text)
data_val["preprocessed_text"] = data_val["preprocessed_text"].apply(lemmatize_text)

## Bag Of Words
Let's get the 10 top words in ham and spam messages (**EXPLORATORY DATA ANALYSIS**)

In [132]:
from sklearn.feature_extraction.text import CountVectorizer

# Separate ham and spam emails
ham_emails = data_train[data_train["label"] == 0]["preprocessed_text"]
spam_emails = data_train[data_train["label"] == 1]["preprocessed_text"]

# Bag of Words for ham
ham_vectorizer = CountVectorizer()
X_ham = ham_vectorizer.fit_transform(ham_emails)

# Bag of Words for spam
spam_vectorizer = CountVectorizer()
X_spam = spam_vectorizer.fit_transform(spam_emails)

In [133]:
# Count total occurrences of each word
ham_word_counts = X_ham.toarray().sum(axis=0)

# Create a DataFrame
ham_df = pd.DataFrame({
    "word": ham_vectorizer.get_feature_names_out(),
    "count": ham_word_counts
})

# Top 10 words
top_10_ham = ham_df.sort_values(by="count", ascending=False).head(10)

print(top_10_ham)

           word  count
4939      state    117
3916         pm     97
5887      would     94
3347         mr     89
4036  president     89
5336       time     81
3838    percent     80
3545      obama     77
813        call     74
4617  secretary     73


In [134]:
spam_word_counts = X_spam.toarray().sum(axis=0)

spam_df = pd.DataFrame({
    "word": spam_vectorizer.get_feature_names_out(),
    "count": spam_word_counts
})

top_10_spam = spam_df.sort_values(by="count", ascending=False).head(10)

print(top_10_spam)

              word  count
9085         money    842
234        account    740
1933          bank    645
5753          fund    625
13733  transaction    466
2429      business    424
3337       country    419
9183            mr    419
8958       million    366
3056       company    363


## Extra features

In [135]:
# We add to the original dataframe two additional indicators (money symbols and suspicious words).
money_simbol_list = "|".join(["euro","dollar","pound","€",r"\$"])
suspicious_words = "|".join(["free","cheap","sex","money","account","bank","fund","transfer","transaction","win","deposit","password"])

data_train['money_mark'] = data_train['preprocessed_text'].str.contains(money_simbol_list)*1
data_train['suspicious_words'] = data_train['preprocessed_text'].str.contains(suspicious_words)*1
data_train['text_len'] = data_train['preprocessed_text'].apply(lambda x: len(x)) 

data_val['money_mark'] = data_val['preprocessed_text'].str.contains(money_simbol_list)*1
data_val['suspicious_words'] = data_val['preprocessed_text'].str.contains(suspicious_words)*1
data_val['text_len'] = data_val['preprocessed_text'].apply(lambda x: len(x)) 

data_train.head()

,text,label,preprocessed_text,money_mark,suspicious_words,text_len
29,"----------- REGARDS, MR NELSON SMITH.KINDLY RE...",1,regard mr nelson smith kindly reply private em...,0,0,79
535,I have not been able to reach oscar this am. W...,0,able reach oscar supposed send pdb receive,0,0,42
695,; Huma Abedin B6I'm checking with Pat on the 5...,0,huma abedin bi checking pat work jack jake res...,0,0,79
557,I can have it announced here on Monday - can't...,0,announced monday today,0,0,22
836,BANK OF AFRICAAGENCE SAN PEDRO14 BP 1210 S...,1,bank africaagence san pedro bp san pedro cote ...,1,1,1103


## How would work the Bag of Words with Count Vectorizer concept?

In [140]:
# Your code

from sklearn.feature_extraction.text import CountVectorizer

# Example corpus
corpus = [
    "free money now",
    "free prize",
    "hello friend"
]

# Create the CountVectorizer
vectorizer = CountVectorizer()

# Generate the Bag of Words matrix
X = vectorizer.fit_transform(corpus)

# Display the vocabulary
print("Vocabulary:")
print(vectorizer.get_feature_names_out())

# Display the Bag of Words matrix
print("\nBag of Words Matrix:")
print(X.toarray())

Vocabulary:
['free' 'friend' 'hello' 'money' 'now' 'prize']

Bag of Words Matrix:
[[1 0 0 1 1 0]
 [1 0 0 0 0 1]
 [0 1 1 0 0 0]]


## TF-IDF

- Load the vectorizer

- Vectorize all dataset

- print the shape of the vetorized dataset

In [142]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Load the vectorizer
tfidf_vectorizer = TfidfVectorizer()

# Vectorize the dataset
X_train_tfidf = tfidf_vectorizer.fit_transform(data_train["preprocessed_text"])

# Print the shape
print(X_train_tfidf.shape)

(800, 20199)


## And the Train a Classifier?

In [144]:
# Your code

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score

# TF-IDF
tfidf_vectorizer = TfidfVectorizer()

X_train = tfidf_vectorizer.fit_transform(data_train["preprocessed_text"])
X_val = tfidf_vectorizer.transform(data_val["preprocessed_text"])

# Labels
y_train = data_train["label"]
y_val = data_val["label"]

# Train classifier
classifier = MultinomialNB()
classifier.fit(X_train, y_train)

# Predict
y_pred = classifier.predict(X_val)

# Accuracy
print("Accuracy:", accuracy_score(y_val, y_pred))

Accuracy: 0.935


### Extra Task - Implement a SPAM/HAM classifier

https://www.kaggle.com/t/b384e34013d54d238490103bc3c360ce

The classifier can not be changed!!! It must be the MultinimialNB with default parameters!

Your task is to **find the most relevant features**.

For example, you can test the following options and check which of them performs better:
- Using "Bag of Words" only
- Using "TF-IDF" only
- Bag of Words + extra flags (money_mark, suspicious_words, text_len)
- TF-IDF + extra flags


You can work with teams of two persons (recommended).

In [139]:
# Your code